In [ ]:
import cv2
import numpy as np

# Load face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(100, 100)
    )

    # Overall frame brightness
    frame_lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    overall_L = np.mean(frame_lab[:, :, 0])

    for (x, y, w, h) in faces:

        # Face box
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        # Center facial region
        roi_x1 = x + int(w * 0.30)
        roi_y1 = y + int(h * 0.30)
        roi_x2 = x + int(w * 0.70)
        roi_y2 = y + int(h * 0.70)

        skin_roi = frame[roi_y1:roi_y2, roi_x1:roi_x2]

        cv2.rectangle(
            frame,
            (roi_x1, roi_y1),
            (roi_x2, roi_y2),
            (255, 0, 0),
            2
        )

        if skin_roi.size > 0:

            lab = cv2.cvtColor(skin_roi, cv2.COLOR_BGR2LAB)

            avg_L = np.mean(lab[:, :, 0])
            avg_A = np.mean(lab[:, :, 1])
            avg_B = np.mean(lab[:, :, 2])

            # Normalize brightness
            normalized_L = avg_L - overall_L + 128

            # Skin tone classification
            if normalized_L > 170:
                tone = "Fair"
            elif normalized_L > 145:
                tone = "Light"
            elif normalized_L > 120:
                tone = "Medium"
            elif normalized_L > 95:
                tone = "Olive"
            else:
                tone = "Deep"

            # Undertone classification
            if avg_B > avg_A + 5:
                undertone = "Warm"
            elif avg_A > avg_B + 5:
                undertone = "Cool"
            else:
                undertone = "Neutral"

            # Display results
            cv2.putText(
                frame,
                f"Tone: {tone}",
                (x, y - 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                f"Undertone: {undertone}",
                (x, y - 15),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 255),
                2
            )

            cv2.putText(
                frame,
                f"L:{avg_L:.1f} A:{avg_A:.1f} B:{avg_B:.1f}",
                (x, y + h + 25),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 0),
                2
            )

    cv2.imshow("Skin Tone & Undertone Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
